# Homework 1 — Agentic RAG over the course lessons

Spec: [cohorts/2026/homework/01-agentic-rag/homework.md](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/homework/01-agentic-rag/homework.md)

The knowledge base switches from the course FAQ (lessons) to the **72 lesson pages themselves**,
pinned to commit `8c1834d` so results are reproducible.

**Decisions taken here**

| Decision | Why |
|---|---|
| Provider Groq, model `openai/gpt-oss-120b` | Free tier. The course uses `gpt-5.4-mini`; gpt-oss is the closest family available on Groq, so token counts land in the same order of magnitude. |
| Subclass `RAGBase` instead of editing `rag_helper.py` | Lesson notebooks keep working; the subclass shows exactly which parts are schema-specific (`search`, `build_context`) and which part changed for usage reporting (`llm`, `rag`). |
| Hand-written agent loop (no `toyaikit`) | Same `chat.completions` tool-calling loop as `agents.ipynb`; no extra dependency and every step is visible. |

## Setup

In [1]:
import json
import os
import time
from dataclasses import dataclass

from dotenv import load_dotenv
from openai import APIStatusError, OpenAI
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

from rag_helper import RAGBase

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "openai/gpt-oss-120b"


def chat(**kwargs):
    """chat.completions.create that waits out Groq's free-tier per-minute token budget.

    Free tier = 8000 tokens/minute, counted across calls. A full-page RAG prompt is ~7k tokens,
    so two calls in the same minute trip it (Groq answers 413 or 429). Waiting resets the window.
    """
    for attempt in range(3):
        try:
            return client.chat.completions.create(model=MODEL, **kwargs)
        except APIStatusError as e:
            if e.status_code not in (413, 429) or "tokens per minute" not in str(e) or attempt == 2:
                raise
            time.sleep(61)

## Data — lesson pages at commit `8c1834d`

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

## Q1. How many lesson pages

In [3]:
len(documents)

72

## Q2. Indexing and searching

`content` is a text field (tokenised, TF-IDF ranked); `filename` is a keyword field (exact-match filter only).

In [4]:
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)
[r["filename"] for r in results]

['01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/15-frameworks.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/11-agents-intro.md',
 '01-agentic-rag/lessons/16-other-frameworks.md']

## Q3. RAG + token usage

`RAGBase` assumes the FAQ schema (`section`/`question`/`answer`). Two methods are schema-specific and get overridden.
`llm` and `rag` are overridden too, so the caller gets the API `usage` object alongside the answer.

In [5]:
@dataclass
class RAGResult:
    answer: str
    usage: object


class LessonsRAG(RAGBase):
    INSTRUCTIONS = """
You're a course teaching assistant. Answer the student's question
using only the provided course lesson pages. If the answer is not
in the context, say you don't know.
""".strip()

    def __init__(self, index, llm_client, model=MODEL, num_results=5):
        super().__init__(index, llm_client, instructions=self.INSTRUCTIONS, model=model)
        self.num_results = num_results

    def search(self, query, num_results=None):
        return self.index.search(query, num_results=num_results or self.num_results)

    def build_context(self, search_results):
        return "\n\n".join(
            f"FILE: {doc['filename']}\n{doc['content']}" for doc in search_results
        )

    def llm(self, prompt):
        return chat(
            messages=[
                {"role": "system", "content": self.instructions},
                {"role": "user", "content": prompt},
            ],
        )

    def rag(self, query):
        prompt = self.build_prompt(query, self.search(query))
        response = self.llm(prompt)
        return RAGResult(response.choices[0].message.content, response.usage)

In [6]:
rag_full = LessonsRAG(index, client)
result_full = rag_full.rag(query)

print(result_full.answer[:800])
print()
print("prompt tokens:", result_full.usage.prompt_tokens)

The agentic loop is just a `while True` that repeatedly sends the current conversation to the model, runs any tool calls that come back, and then checks whether the model asked for another tool.  

**How it works (from 14‑agentic‑loop.md)**  

1. **Start the loop** – an iteration counter (`it`) is initialized and the loop begins:  

   ```python
   while True:
       print(f"iteration #{it}...")
       has_function_calls = False
   ```

2. **Call the model** – the current `messages` list (developer prompt, user question, plus every previous model output and tool result) is sent to the LLM:

   ```python
   response = openai_client.responses.create(
       model="gpt-5.4-mini",
       input=messages,
       tools=[search_tool],
   )
   ```

3. **Add the model’s output to memory** – `respons

prompt tokens: 7176


## Q4. Chunking

Sliding window: 2000-char windows, moving 1000 chars at a time, so neighbours overlap by 1000 chars.
A passage cut at one boundary appears whole in the next window.

In [7]:
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

## Q5. RAG with chunking

In [8]:
chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

rag_chunks = LessonsRAG(chunk_index, client)
result_chunks = rag_chunks.rag(query)

full_tokens = result_full.usage.prompt_tokens
chunk_tokens = result_chunks.usage.prompt_tokens
print("full pages:", full_tokens, "| chunks:", chunk_tokens, f"| ratio: {full_tokens / chunk_tokens:.1f}x")

full pages: 7176 | chunks: 2359 | ratio: 3.0x


## Q6. Turning it into an agent

Same loop as `agents.ipynb`: call the model with the tool schema; while it returns `tool_calls`,
execute them, append results as `role: tool` messages, call again. Stop when it returns plain text
(or after `MAX_ITERATIONS`, a safety cap against runaway loops).

The tool schema is written by hand here. Frameworks like toyaikit or PydanticAI generate the same JSON
from the function's type hints and docstring, which is why the homework asks for both.

**Context budget.** Every tool result stays in `messages`, so each search adds ~2.5k tokens (5 chunks x 2000 chars).
Groq's free tier caps a single request at 8000 tokens/minute, so with 5 results the 4th call failed with
`413 Request too large (Requested 8591)`. Two changes keep the agent within budget:
the tool returns 3 chunks and only the fields the model needs (`filename`, `content`, not `start`),
and `chat()` waits out the per-minute window when consecutive calls exhaust it.
Production agents face the same problem at a larger scale: truncating or summarising tool output is part of the design.

In [9]:
AGENT_NUM_RESULTS = 3  # free-tier budget, see note above


def search(query: str) -> list[dict]:
    """Search the LLM Zoomcamp lesson pages for chunks matching the query."""
    results = chunk_index.search(query, num_results=AGENT_NUM_RESULTS)
    return [{"filename": r["filename"], "content": r["content"]} for r in results]


search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": search.__doc__,
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "Search keywords."}},
            "required": ["query"],
            "additionalProperties": False,
        },
    },
}

AGENT_INSTRUCTIONS = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)
MAX_ITERATIONS = 10



def run_agent(question):
    messages = [
        {"role": "system", "content": AGENT_INSTRUCTIONS},
        {"role": "user", "content": question},
    ]
    search_calls = []

    for _ in range(MAX_ITERATIONS):
        response = chat(messages=messages, tools=[search_tool])
        message = response.choices[0].message

        if not message.tool_calls:
            return message.content, search_calls

        messages.append(message)
        for call in message.tool_calls:
            args = json.loads(call.function.arguments)
            search_calls.append(args["query"])
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(search(**args)),
            })

    return None, search_calls

In [10]:
answer, search_calls = run_agent(
    "How does the agentic loop work, and how is it different from plain RAG?"
)

print("search calls:", len(search_calls))
for q in search_calls:
    print(" -", q)
print()
print(answer)

search calls: 3
 - agentic loop
 - plain RAG
 - agentic loop vs rag

**The agentic loop vs. plain RAG**

| Aspect | Plain Retrieval‑Augmented Generation (RAG) | Agentic (tool‑using) Loop |
|--------|--------------------------------------------|----------------------------|
| **Core idea** | *Retrieve* a set of relevant passages, *augment* the prompt with them, then call the LLM **once** to generate the answer. | A **while‑loop** that repeatedly calls the LLM, lets the model request *tools* (search, calculator, code execution, etc.), runs those tools, feeds the results back, and continues until the model signals “I’m done.” |
| **Control flow** | Fixed three‑step pipeline:<br>1️⃣ search → 2️⃣ prompt construction → 3️⃣ LLM generation. No further interaction. | Dynamic, multi‑turn interaction: <br>```\nwhile True:\n    response = llm(messages)\n    if response contains tool calls:\n        run each tool, add tool‑output messages\n    else:\n        break   # final answer\n``` |
| **Tool u